# Differential chromatic refraction for DDF

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from scipy import stats


from speclite import filters

# from specutils.utils.wcs_utils import refraction_index
import ref_index


from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u
from astropy.timeseries import TimeSeries
from astropy.coordinates import get_sun


from astroplan import Observer
from astroplan import FixedTarget
from astroplan.plots import plot_airmass, plot_parallactic, plot_altitude, plot_sky
from astroplan import is_observable

from pytz import timezone


from scipy.integrate import simpson

warnings.filterwarnings("ignore")
print(f"pandas   version : {pd.__version__}")
print(f"numpy    version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
def zenith_tangent_vector(ra_deg, dec_deg, obstime, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith

    Parameters:
    ==========
        ra_deg,dec_deg: target coordinates in the sky
        obstimes:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
    """

    # Source
    sky = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg)

    # Zénith en AltAz → (alt=90°, az arbitraire)
    zenith_altaz = SkyCoord(alt=90 * u.deg, az=0 * u.deg, frame=AltAz(obstime=obstime, location=location))

    # Convertir en ICRS
    zenith_icrs = zenith_altaz.transform_to("icrs")

    # Vecteurs cartésiens
    s = sky.cartesian.xyz.value
    z = zenith_icrs.cartesian.xyz.value

    # Projection tangentielle
    v = z - np.dot(z, s) * s

    # Normalisation
    v /= np.linalg.norm(v)

    return v  # vecteur 3D tangent au ciel

In [ ]:
def zenith_tangent_vector_fromHA(HA_deg, coords, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith
    Parameters:
    ==========
        HA_deg : array of Hour angles
        coords: target SkyCoords
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
        array if sinz values (related to dipole intensity)
    """

    lat_deg = location.lat.to(u.deg).value

    dec_deg = coords.dec.to(u.deg).value
    ra_deg = coords.ra.to(u.deg).value

    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    # --- direction source ---
    s = np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])  # (3,)

    # --- zénith ---
    HA_val = HA_deg.to(u.deg).value  # ← FIX unités
    lst = np.deg2rad(HA_val + ra_deg)
    lat = np.deg2rad(lat_deg)

    z = np.array(
        [np.cos(lat) * np.cos(lst), np.cos(lat) * np.sin(lst), np.sin(lat) * np.ones_like(lst)]
    )  # (3, N)

    # --- projection ---
    # v = z - np.dot(z, s) * s
    proj = np.sum(z * s[:, None], axis=0)  # (N,)
    v = z - proj * s[:, None]  # (3, N)

    # --- norme par point ---
    norm = np.linalg.norm(v, axis=0)  # (N,)

    # --- normalisation optionnelle ---
    v_unit = np.zeros_like(v)
    mask = norm > 0
    v_unit[:, mask] = v[:, mask] / norm[mask]

    return v_unit, norm

On note :

- H : angle horaire
- δ : déclinaison de la source
- ϕ : latitude de l’observatoire
- z : angle zénithal
### 1) Cosinus de l’angle zénithal

$$\cos z = \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H$$

### 2) Donc sin(z)
$$\sin z = \sqrt{1 - \left(\sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H\right)^2}$$	

✔ Version compacte équivalente (souvent plus utile)

On passe par l’altitude h :

$$\sin h = \sin\phi\sin\delta + \cos\phi\cos\delta\cos H$$

et :

$$z = \frac{\pi}{2} - h$$

donc :

$$\sin z = \cos h$$

✔ Formule finale (celle de ta fonction Python)

$$\boxed{ \sin z(H,\delta,\phi) = \sqrt{ 1 - \left( \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H \right)^2 } }$$

✔ Interprétation physique (très important pour ton dipôle)
H → rotation du ciel
ϕ → géométrie de l’observatoire
δ → position du champ

👉 donc :

- amplitude du dipôle ∝ sinz(H)
- orientation du dipôle ∝ parallactic angle q(H)

In [ ]:
def sinz_vs_HA(HA_deg, coords, location):
    """
    Compute the sinus of zenith angle from the formula
    \sin z(H,\delta,\phi) = \sqrt{ 1 - \left( \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H \right)^2

    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree

    """

    # --- location ---> latitude
    lat_deg = location.lat.to(u.deg).value

    # --- object ---> declination
    dec_deg = coords.dec.to(u.deg).value

    # --- HA be sure to have quanitites in deg
    HA_valdeg = HA_deg.to(u.deg).value

    HA = np.deg2rad(HA_valdeg)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    return np.sqrt(1 - cosz**2)

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle(coords, times, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # LST
    lst = times.sidereal_time("apparent", longitude=location.lon)

    # angle horaire H = LST - RA
    H = (lst - coords.ra).to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(H)
    cosH = np.cos(H)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle_fromHA(ha, coords, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        ha:  hour angle Angle in degree
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # angle horaire H = LST - RA
    ha_rad = ha.to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(ha_rad)
    cosH = np.cos(ha_rad)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

## Initialisation

In [ ]:
# ── I/O paths ─────────────────────────────────────────────────────────────────
NB_TAG = "TOOLS_07_DDF-DCR"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Figures: {os.path.abspath(DIR_FIGS)}")

### Target initialisations

In [ ]:
# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

DEEP_FIELDS_COLORSTYLE = {
    "COSMOS": {"color": "r"},
    "ELAIS-S1": {"color": "k"},
    "ECDFS": {"color": "grey"},
    "EDFS-a": {"color": "b"},
    "EDFS-b": {"color": "g"},
    "EDFS": {"color": "magenta"},
    "M49": {"color": "purple"},
}

In [ ]:
DDF_NAMES = list(DEEP_FIELDS.keys())  # fixed order for 2×3 grid

# ── Band colours (LSST ugrizy) ────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

# ── Matplotlib defaults ───────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure as PDF + PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print("Configuration done.")

### Observatory initialisation

In [ ]:
# ── Rubin/LSST observatory location (Cerro Pachón) ───────────────────────────
RUBIN_LAT_DEG = -30.244728  # degrees North
RUBIN_LON_DEG = -70.749417  # degrees East  (West is negative)
RUBIN_HEIGHT_M = 2647.0  # metres above sea level

RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Rubin/LSST : lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

### observer in astroplan in case one want to check

In [ ]:
observer = Observer.at_site("lsst", timezone="UTC")

In [ ]:
observer

In [ ]:
def zenith_tangent_vector_fromHA(HA_deg, coords, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith
    Parameters:
    ==========
        HA_deg : array of Hour angles
        coords: target SkyCoords
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
        array if sinz values (related to dipole intensity)
    """

    lat_deg = location.lat.to(u.deg).value

    dec_deg = coords.dec.to(u.deg).value
    ra_deg = coords.ra.to(u.deg).value

    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    # --- direction source ---
    s = np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])  # (3,)

    # --- zénith ---
    HA_val = HA_deg.to(u.deg).value  # ← FIX unités
    lst = np.deg2rad(HA_val + ra_deg)
    lat = np.deg2rad(lat_deg)

    z = np.array(
        [np.cos(lat) * np.cos(lst), np.cos(lat) * np.sin(lst), np.sin(lat) * np.ones_like(lst)]
    )  # (3, N)

    # --- projection ---
    # v = z - np.dot(z, s) * s
    proj = np.sum(z * s[:, None], axis=0)  # (N,)
    v = z - proj * s[:, None]  # (3, N)

    # --- norme par point ---
    norm = np.linalg.norm(v, axis=0)  # (N,)

    # --- normalisation optionnelle ---
    v_unit = np.zeros_like(v)
    mask = norm > 0
    v_unit[:, mask] = v[:, mask] / norm[mask]

    return v_unit, norm

In [ ]:
def sinz_vs_HA(HA_deg, coords, location):
    """
    Compute the sinus of zenith angle from the formula
    \sin z(H,\delta,\phi) = \sqrt{ 1 - \left( \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H \right)^2

    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree

    """

    # --- location ---> latitude
    lat_deg = location.lat.to(u.deg).value

    # --- object ---> declination
    dec_deg = coords.dec.to(u.deg).value

    # --- HA be sure to have quanitites in deg
    HA_valdeg = HA_deg.to(u.deg).value

    HA = np.deg2rad(HA_valdeg)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    return np.sqrt(1 - cosz**2)

In [ ]:
def tanz_vs_HA(HA_deg, coords, location):
    """
    Compute the tan of zenith angle from the formula
    \sin z(H,\delta,\phi) = \sqrt{ 1 - \left( \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H \right)^2

    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree

    """

    # --- location ---> latitude
    lat_deg = location.lat.to(u.deg).value

    # --- object ---> declination
    dec_deg = coords.dec.to(u.deg).value

    # --- HA be sure to have quanitites in deg
    HA_valdeg = HA_deg

    HA = np.deg2rad(HA_valdeg)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    sinz = np.sqrt(1 - cosz**2)
    tanz = sinz / cosz

    return tanz

## Refractive index in LSST bands

In [ ]:
def pivot_lambda(weight, lam):
    return np.sum(weight * lam) / np.sum(weight)

In [ ]:
# -----------------------------
# 1. LSST filters (speclite)
# -----------------------------
bands = ["u", "g", "r", "i", "z", "y"]
lsst_seq = filters.load_filters("lsst2023-*")
lsst = {f.name: f for f in lsst_seq}

In [ ]:
# -----------------------------
# 2. Wavelength grid (microns)
# -----------------------------
lam = np.linspace(0.3, 1.1, 2000)  # LSST range in µm

In [ ]:
# -----------------------------
# 3. AB flat spectrum
# f_nu constant => f_lambda ∝ λ^-2
# -----------------------------
f_lambda = lam ** (-2)

In [ ]:
# -----------------------------
# 4. Simple Ciddor-like refractive index
# -----------------------------
def n_of_lambda(lam, t=20, p=101325, rh=20):
    """
    provide the refraction index  using the Ciddor formula
    Parameters:
    ==========
         lam : array of wavelengths in microns
    Returns:
    ========
         refraction array
    """
    return ref_index.ciddor(wave=lam * 1000, t=t, p=p, rh=rh)


n_lam = n_of_lambda(lam)

In [ ]:
# -----------------------------
# 5. Reference wavelength (r band effective)
# -----------------------------
# lam_ref = 0.62  # µm (r-band pivot)
# n_ref = n_of_lambda(lam_ref)
n_ref = 1.0

In [ ]:
# -----------------------------
# 6. Atmospheric geometry
# -----------------------------
def refraction_shift(z_deg, lam):
    """

    Parameters:
    ==========
         z_deg : zenith angle in degree (scalar)
         lam : array of wavelengths in microns
    Returns:
    ========
         refraction array shift
    """
    z = np.deg2rad(z_deg)
    n_lam = n_of_lambda(lam)  # tableau !
    return (n_lam - n_ref) * np.tan(z)

In [ ]:
# -----------------------------
# 6. Atmospheric geometry
# -----------------------------
def refraction_index(lam):
    """
    Parameters:
    ==========
         lam : array of wavelengths in microns
    Returns:
    ========
         refraction array shift
    """
    n_lam = n_of_lambda(lam)  # tableau !
    return n_lam - n_ref

### 1. Ce que tu modélises réellement

Tu simules une réfraction atmosphérique chromatique, donc :
$$R(\lambda,\theta_z)$$

où :

- $\lambda$ → dispersion (indice de réfraction)
- $\theta_z$ → géométrie du trajet dans l’atmosphère (angle zenithal

### 2. Pourquoi tan z apparaît

Dans l’approximation standard LSST :

- atmosphère plane-parallèle
- faible réfraction
- déviation intégrée le long du trajet

on obtient :

$$ R(\lambda,\theta_z)≈(n(\lambda)−1)\tan \theta_z $$

👉 $\tan \theta_z$  vient de la géométrie du chemin optique dans une atmosphère stratifiée

In [ ]:
# -----------------------------
# 7. Band-averaged DCR
# -----------------------------
def band_dcr(band, z_deg):

    bp = lsst[f"lsst2023-{band}"]
    lam_bp = bp.wavelength * 1e-4  # Å → µm

    trans = np.interp(lam, lam_bp, bp.response, left=0.0, right=0.0)

    weight = trans * f_lambda * lam
    weight /= simpson(weight, lam)

    # 👉 REFRACTION CONTINUE
    dR_lam = refraction_shift(z_deg, lam)

    # 👉 INTÉGRATION CORRECTE
    dR_band = simpson(weight * dR_lam, lam)

    return dR_band

In [ ]:
# -----------------------------
# 7. Band-averaged DCR
# -----------------------------
def band_nindex_dcr(band):

    bp = lsst[f"lsst2023-{band}"]
    lam_bp = bp.wavelength * 1e-4  # Å → µm

    trans = np.interp(lam, lam_bp, bp.response, left=0.0, right=0.0)

    weight = trans * f_lambda * lam
    weight /= simpson(weight, lam)

    # 👉 REFRACTION CONTINUE
    dN_lam = refraction_index(lam)

    # 👉 INTÉGRATION CORRECTE
    dN_band = simpson(weight * dN_lam, lam)

    return dN_band

In [ ]:
# -----------------------------
# 8. Parallactic angle projection
# -----------------------------
def project_dcr(delta_theta, q_deg):
    q = np.deg2rad(q_deg)
    return {"RA": delta_theta * np.cos(q), "Dec": delta_theta * np.sin(q)}

In [ ]:
# -----------------------------
# 9. Airmass helper
# -----------------------------
def airmass_to_z(X):
    z = np.arccos(1 / X)
    return np.rad2deg(z)

In [ ]:
# -----------------------------
# 10. Example computation
# -----------------------------
X = 1.5
q = 30  # parallactic angle
rad_to_mas = 180.0 / np.pi * 3600 * 1000
rad_to_arcsec = 180.0 / np.pi * 3600
z = airmass_to_z(X)

results = {}

for b in bands:
    dtheta = band_dcr(b, z)  # radians
    proj = project_dcr(dtheta, q)

    results[b] = {
        "dtheta_mas": dtheta * rad_to_arcsec,
        "RA_mas": proj["RA"] * rad_to_arcsec,
        "Dec_mas": proj["Dec"] * rad_to_arcsec,
    }

for b, v in results.items():
    print(b, v)

In [ ]:
def plotDCR(ha, coords, location, ax, ha_min=-90, ha_max=90, ha_unit="deg", field_name=None):
    """ """
    rad_to_arcsec = 180.0 / np.pi * 3600

    # geometric effect
    if ha_unit == "deg":
        tanz = tanz_vs_HA(ha, coords, location)
    elif ha_unit == "hour":
        tanz = tanz_vs_HA(ha * 15.0, coords, location)
    else:
        raise ("unknown ha unit {ha_unit}")

    # refraction index
    dN = {}
    dR = {}
    for b in bands:
        dN[b] = band_nindex_dcr(b)
        dR[b] = rad_to_arcsec * dN[b] * tanz
        ax.plot(ha, dR[b], color=BAND_COLORS[b])

In [ ]:
location = RUBIN_LOCATION
ha = np.linspace(-4, 4, 100)
ha_unit = "hour"

for field_name in DDF_NAMES:
    ra_deg = DEEP_FIELDS[field_name][0]
    dec_deg = DEEP_FIELDS[field_name][1]
    coords = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg)

    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    plotDCR(ha, coords, location, ax, ha_min=-6, ha_max=6, ha_unit="hour", field_name=None)
    ax.set_title(field_name)
    # ax.set_yscale("log")
    fig.show()